In [ ]:
from pymongo import MongoClient
import requests
import json
import pandas as pd
from thefuzz import process
import re

In [ ]:
res = requests.get("http://110.164.203.137:9919/anime")
data = res.json() 
anime = pd.DataFrame(data)
comment = pd.read_json('anime_2025_thread_comments_new.json')

In [ ]:
# 1. ฟังก์ชันล้างคำส่วนเกิน (Cleaning) เพื่อให้ Match ง่ายขึ้น
def clean_anime_name(text):
    if pd.isna(text): return ""
    text = text.lower()
    # ลบคำที่มักจะทำให้การ Match เพี้ยน
    text = re.sub(r'season \d+|episode of|the animation|:', '', text)
    # ลบช่องว่างส่วนเกิน
    return " ".join(text.split())

# 2. เตรียมข้อมูล (สมมติว่าคุณมี comment และ anime อยู่แล้ว)
# สร้างคอลัมน์ชั่วคราวสำหรับเทียบ (Cleaned Name)
comment['clean_name'] = comment['anime'].apply(clean_anime_name)
anime['clean_title'] = anime['title'].apply(clean_anime_name)

In [ ]:
# 3. สร้าง Dictionary เพื่อเก็บผลการ Match (กันการคำนวณซ้ำเพื่อความเร็ว)
unique_comments = comment['clean_name'].unique()
choices = anime['clean_title'].tolist()
match_dict = {}

print("กำลังวิเคราะห์ความคล้ายของชื่อเรื่อง... (Fuzzy Matching)")

for name in unique_comments:
    # หาชื่อที่ใกล้เคียงที่สุด (Threshold 80% คือกำลังดี)
    best_match, score = process.extractOne(name, choices)
    if score >= 80:
        # หา Index ของชื่อที่ Match ติดในตาราง anime เพื่อดึงชื่อ 'ต้นฉบับ' มา
        original_title = anime.iloc[choices.index(best_match)]['title']
        match_dict[name] = original_title
    else:
        match_dict[name] = None


In [ ]:
# 4. นำผลลัพธ์ที่ Match ได้ กลับไปใส่ในตาราง comment
comment['matched_title'] = comment['clean_name'].map(match_dict)

# 5. Merge ข้อมูลเข้าด้วยกัน
df_merged = pd.merge(comment, anime, left_on='matched_title', right_on='title', how='left')

# แสดงผลลัพธ์ที่ Match ติด
matched_count = df_merged['title'].notna().sum()
print(f"--- สรุปผล ---")
print(f"จับคู่สำเร็จ: {matched_count} แถว")
print(df_merged[['anime', 'title']].sample(10)) # ดูตัวอย่างการจับคู่

In [ ]:
cols_to_drop = ['clean_name', 'clean_title', 'matched_title']
df_final = df_merged.drop(columns=[c for c in cols_to_drop if c in df_merged.columns])

# 2. ตรวจสอบดูว่าเหลือค่าว่าง (NaN) อีกกี่แถว
print("จำนวนค่าว่างคงเหลือในคอลัมน์ title:", df_final['title'].isna().sum())

In [ ]:
# กรองเฉพาะแถวที่คอลัมน์ 'title' (จากฝั่งขวา) เป็นค่าว่าง
missing_data = df_merged[df_merged['title'].isnull()]

# แสดงรายชื่ออนิเมะที่ไม่ซ้ำกันที่ Merge ไม่ติด
print("รายชื่ออนิเมะที่ยังว่างอยู่ (หาคู่ไม่เจอ):")
print(missing_data['anime'].unique())